# v07 SFT — Qwen3.5-4B (Unsloth QLoRA, code-gen PoT)

Fine-tune Qwen3.5-4B to emit **5–10 line reasoning + ONE Python code block** that prints
`FINAL ANSWER:` / `UNIT:`. Validated on Colab **A100-40GB**.

Pipeline: setup env → (data already built) → SFT → merge+push to Hub → eval on val_56 + 60 golden.

> Deps live in an isolated venv `/content/v07_env`; train/eval run with `/content/v07_env/bin/python`.


## 1. CUDA paths (run once) — then `nvidia-smi` works


In [ ]:
!echo 'export LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/nvidia/lib64:$LD_LIBRARY_PATH' >> ~/.bashrc
!echo 'export PATH=/usr/local/nvidia/bin:/usr/local/cuda/bin:$PATH' >> ~/.bashrc
import os
os.environ['LD_LIBRARY_PATH']='/usr/lib64-nvidia:/usr/local/nvidia/lib64:'+os.environ.get('LD_LIBRARY_PATH','')
os.environ['PATH']='/usr/local/nvidia/bin:/usr/local/cuda/bin:'+os.environ['PATH']
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


## 2. Get the repo onto `/content`
Private repo → clone with a GitHub token (set `GH_TOKEN`). Branch `Nguyen/submition_v1`.
(If you already SSH-copied the repo to `/content/Exact_2026_Laplace-s_Red_Devils`, skip this.)


In [ ]:
GH_TOKEN = ''  # <-- fill in (repo is private)
REPO = '/content/Exact_2026_Laplace-s_Red_Devils'
import os
if not os.path.exists(REPO):
    !git clone -b Nguyen/submition_v1 https://{GH_TOKEN}@github.com/fishperson113/Exact_2026_Laplace-s_Red_Devils.git {REPO}
os.chdir(REPO); print('cwd:', os.getcwd())


## 3. Install the proven stack (isolated venv)
Creates `/content/v07_env` with torch 2.10 + transformers 5.5 + unsloth 2026.5.6. ~several min, ~10 GB.


In [ ]:
!bash app/physics_solution/versions/v07_final_version/colab_setup/setup_colab.sh
VENV='/content/v07_env/bin/python'


## 4. (Optional) Rebuild data
`train.jsonl`/`val.jsonl` are committed. Re-run only to regenerate (rebuilds val_56 + splits, golden held out).


In [ ]:
%env PYTHONPATH=/content/Exact_2026_Laplace-s_Red_Devils
# !$VENV -m app.physics_solution.versions.v07_final_version.make_val
# !$VENV -m app.physics_solution.versions.v07_final_version.build_sft
!wc -l app/physics_solution/versions/v07_final_version/output/train.jsonl app/physics_solution/versions/v07_final_version/output/val.jsonl


## 5. HF token (for merge + push to Hub)


In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # <-- Laplaces-Red-Devils write token
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']


## 6. Train (Unsloth QLoRA)
Config: `train/configs/sft.yaml` (r=16, α=32, 3 epochs, eff-batch 16, train-on-completion).
Saves adapter+tokenizer, merges, pushes **adapter** + **merged** repos with a metrics model card.
Add `--skip-hub` to train without pushing.


In [ ]:
%env PYTHONPATH=/content/Exact_2026_Laplace-s_Red_Devils
!$VENV -m app.physics_solution.versions.v07_final_version.train.train \
  --config app/physics_solution/versions/v07_final_version/train/configs/sft.yaml


## 7. Eval — val_56 + 60 golden
Generate → extract code → execute sandboxed → score `FINAL ANSWER:`/`UNIT:`. Reports both sets.
Point `--model` at the local merged dir or the pushed merged repo.


In [ ]:
%env PYTHONPATH=/content/Exact_2026_Laplace-s_Red_Devils
MERGED='app/physics_solution/versions/v07_final_version/train/runs/sft_v07/merged'
!$VENV -m app.physics_solution.versions.v07_final_version.eval --model $MERGED


## Notes / troubleshooting
- `Qwen3_5 does not support SDPA — switching to fast eager`: expected (Unsloth), harmless.
- `Flash Attention 2 ... broken. Using Xformers`: expected; no perf loss for this size.
- fla/causal-conv1d not required (torch fallback). `setup_colab.sh` tries them best-effort.
- OOM on A100-40G: lower `per_device_train_batch_size` or `max_seq_length` in `sft.yaml`.
- Base weights (~8 GB) download once, then cached under the HF cache.
